In [ ]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END 
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
import os
from pathlib import Path

openai_llm = ChatOpenAI(
    model="gpt-4.1-mini"
)

In [9]:
class QAstate(TypedDict):
    question: Optional[str]
    context: Optional[str]
    answer : Optional[str]

def input_validation_node(state):
    question = state.get("question", "").strip()
    if not question:
        return {"valid": False , "error": "Question should not be empty!"}
    return {"valid": True}

def context_provider_node(state):
    question = state.get("question", "").lower()
    if "langgraph" in question or "guided project" in question:
        context=(
            "This guided project is about using LangGraph, a Python library to design state-based workflows. "
            "LangGraph simplifies building complex applications by connecting modular nodes with conditional edges."
        )
        return {"context": context}
    return {"context" : None}

def llm_qa(state):
    question = state.get("question", "")
    context = state.get("context", None)
    if not context:
        return {"context":"i dont have context related to question"}
    prompt = f"Context: {context}\n Question: {question} \n Answer the question based on provided context."
    try: 
        response = openai_llm.invoke(prompt)
        return {"answer": response.content.strip()}
    except Exception as e:
        return {"answer": f"An error ocured: {str(e)}"}
workflow = StateGraph(QAstate)
workflow.add_node("Inputnode", input_validation_node)
workflow.add_node("context", context_provider_node)
workflow.add_node("llm", llm_qa)
workflow.set_entry_point("Inputnode")
workflow.add_edge("Inputnode", "context")
workflow.add_edge("context", "llm")
workflow.add_edge("llm", END)
app= workflow.compile()
output = app.invoke({"question": "What is langgraph?"})
print(output)

{'question': 'What is langgraph?', 'context': 'This guided project is about using LangGraph, a Python library to design state-based workflows. LangGraph simplifies building complex applications by connecting modular nodes with conditional edges.', 'answer': "An error ocured: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}"}
